<a href="https://colab.research.google.com/github/eodenyire/LearnDataScienceWithEmmanuelOdenyire/blob/main/Phase_3_Advanced_AI_and_Deployment/Month_12_End_to_End_Projects/Week_4_MLOps_and_Monitoring/Week_4_Model_Monitoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 4: Model Monitoring and Maintenance

## Learning Objectives
- Detect data drift using statistical tests
- Monitor model performance over time
- Implement alerting for performance degradation
- Build a monitoring dashboard

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

rng = np.random.default_rng(42)
sns.set_theme(style='whitegrid')

## 1. Data Drift Detection

In [ ]:
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_prod_clean = X_test.copy()
# Simulate distribution shift in production data
X_prod_drift = X_test.copy()
X_prod_drift['MedInc'] = X_prod_drift['MedInc'] * 1.5 + rng.normal(0, 0.5, len(X_prod_drift))

print('KS Test for Data Drift:')
for col in X.columns:
    stat, p = stats.ks_2samp(X_prod_clean[col], X_prod_drift[col])
    drift = 'DRIFT' if p < 0.05 else 'OK'
    print(f'  {col:15s}: KS={stat:.4f}, p={p:.4f} [{drift}]')

## 2. Population Stability Index (PSI)

In [ ]:
def psi(expected, actual, bins=10):
    expected_pcts, edges = np.histogram(expected, bins=bins, density=True)
    actual_pcts, _       = np.histogram(actual,   bins=edges, density=True)
    expected_pcts = np.clip(expected_pcts, 1e-4, None)
    actual_pcts   = np.clip(actual_pcts,   1e-4, None)
    psi_value = np.sum((actual_pcts - expected_pcts) * np.log(actual_pcts / expected_pcts))
    return psi_value

print('PSI Values:')
for col in X.columns:
    p = psi(X_prod_clean[col], X_prod_drift[col])
    label = 'STABLE' if p < 0.1 else ('SLIGHT' if p < 0.2 else 'SIGNIFICANT')
    print(f'  {col:15s}: PSI={p:.4f} [{label}]')

## 3. Performance Monitoring Over Time

In [ ]:
model = RandomForestRegressor(n_estimators=50, random_state=42)
model.fit(X_train, y_train)

# Simulate monthly performance slippage
months = 12
base_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
print(f'Baseline RMSE: {base_rmse:.4f}')

monthly_rmses = []
for month in range(months):
    noise = rng.normal(0, 0.02 * month, len(X_test))
    y_noisy = y_test + noise
    rmse = np.sqrt(mean_squared_error(y_noisy, model.predict(X_test)))
    monthly_rmses.append(rmse)

plt.figure(figsize=(10, 4))
plt.plot(range(1, months+1), monthly_rmses, marker='o')
plt.axhline(base_rmse, color='green', linestyle='--', label='Baseline')
plt.axhline(base_rmse * 1.2, color='red', linestyle='--', label='Alert Threshold')
plt.title('Monthly RMSE Over Time')
plt.xlabel('Month'); plt.ylabel('RMSE')
plt.legend(); plt.tight_layout(); plt.show()

## Exercise

1. Implement Evidently AI for automated drift reports
2. Set up a Prometheus metrics endpoint for a Flask model API
3. Build a simple anomaly detector for incoming predictions

In [ ]:
# Your code here


## Summary

- ✓ KS test for feature drift detection
- ✓ Population Stability Index (PSI)
- ✓ Performance monitoring over time
- ✓ Alerting thresholds

## Next Week
MLOps Introduction!